In [1]:
from pathlib import Path

import prism

from imagematerials.eol import eol_preprocess
from imagematerials.factory import ModelFactory, Sector
from imagematerials.model import (
    EndOfLife,
    GenericMaterials,
    GenericStocks,
    Maintenance,
    MaterialIntensities,
    RestOf
)
from imagematerials.preprocessing import get_preprocessing_data
import numpy as np

from imagematerials.rest_of import rest_of_preprocessing
import matplotlib.pyplot as plt

from imagematerials.electricity.preprocessing import (
    get_preprocessing_data_gen,
    get_preprocessing_data_grid)


In [ ]:
scenario_name = "SSP2_M_CP"

In [3]:
climate_policy_scenario_dir = Path("..", "data", "raw", "image", scenario_name)
scenario_base_path = Path("../data/raw") / 'circular_economy_scenarios'

path_image_materials = Path("C:/Coding/image-materials")

raw_data = path_image_materials / "data/raw"

In [4]:
#load historic data
from imagematerials.rest_of.resource_model import ResourceModel

steel = ResourceModel(resource_group = 'metals', resource = 'steel', 
                        image_mat_available = True, start_year = 1971,
                        scenario=scenario_name,
                        convert_image=True, end_year = 2012, convert_to_tons = 1/1000_000, 
                        trade_data=True, path_input_data="../data/raw/rest-of", 
                        path_input_data_image = Path("../data/raw/image"))

aluminium = ResourceModel(resource_group = 'metals', resource = 'aluminium', 
                        image_mat_available = True, start_year = 1998, 
                        scenario=scenario_name, end_year = 2024, path_input_data="../data/raw/rest-of", 
                        path_input_data_image = Path("../data/raw/image")
                        )

cement = ResourceModel(resource_group = 'nmm', resource = 'cement', 
                    image_mat_available = True, start_year = 1971, 
                    scenario=scenario_name,
                    convert_image=True, end_year = 2012, convert_to_tons = 1/1000_000, 
                    trade_data=True, path_input_data="../data/raw/rest-of", 
                    path_input_data_image = Path("../data/raw/image"))

copper = ResourceModel(resource_group = 'metals', resource = 'copper', 
                       image_mat_available = True, start_year = 1990,
                       scenario= scenario_name, end_year = 2011,
                       path_input_data="../data/raw/rest-of", 
                       path_input_data_image = Path("../data/raw/image"))

sand = ResourceModel(resource_group = 'nmm', resource = 'sand_gravel_crushed_rock', 
                       image_mat_available = True, start_year = 1970, scenario= scenario_name, 
                       path_input_data="../data/raw/rest-of", 
                    path_input_data_image = Path("../data/raw/image"))

limestone = ResourceModel(resource_group = 'nmm', resource = 'limestone', 
                    image_mat_available = False, start_year = 1970, scenario=scenario_name, 
                    path_input_data="../data/raw/rest-of", 
                    path_input_data_image = Path("../data/raw/image"))

clay = ResourceModel(resource_group = 'nmm', resource = 'clays', 
                    image_mat_available = False, start_year = 1970, 
                    scenario=scenario_name, 
                    path_input_data="../data/raw/rest-of", 
                    path_input_data_image = Path("../data/raw/image"))



In [5]:


# Define the complete timeline, including historic tail
# time_start = prep_data["stocks"].coords["Time"].min().values
time_start = 1960
complete_timeline = prism.Timeline(time_start, 2060, 1)
simulation_timeline = prism.Timeline(1970, 2060, 1)


bld_sector = get_preprocessing_data("buildings", Path("..", "data", "raw"), 
                                    climate_policy_scenario_dir, 
                                    circular_economy_scenario_dirs = None) 
vhc_sector = get_preprocessing_data("vehicles", raw_data, 
                                    climate_policy_scenario_dir, 
                                    circular_economy_scenario_dirs = None)
rest_sector = rest_of_preprocessing(raw_data, 
                    image_scenario_directory = climate_policy_scenario_dir)

# needed for electricity model
scen = scenario_name.split("_")[0]
variant = "_".join(scenario_name.split("_")[1:])

# TODO fix this for real in the future
prep_data_vhc = vhc_sector.prep_data
prep_data_el = get_preprocessing_data_gen(path_image_materials, scen, variant)
_, prep_data_grid = get_preprocessing_data_grid(path_image_materials, scen, variant)


vhc_sector = Sector('vehicles', prep_data_vhc)
rest_sector = Sector(name='rest_of', 
                    data = rest_sector,)
sec_electr_gen = Sector("generation", prep_data_el)
sec_electr_grid = Sector("grid", prep_data_grid)

factory = ModelFactory(
[bld_sector, vhc_sector, rest_sector, sec_electr_gen, sec_electr_grid], complete_timeline
).add(GenericStocks, ["buildings", "vehicles", "generation", "grid"]
).add(GenericMaterials,  "vehicles"
).add(MaterialIntensities, ["buildings", "generation", "grid"]
).add(RestOf, "rest_of", input_sources={
    "gompertz_coefs": "rest_of",
    "gdp_per_capita": "rest_of",
    "population": "rest_of",
    "historic_diff_consumption": "rest_of"
}
)
model = factory.finish()

import warnings
with warnings.catch_warnings():
    warnings.filterwarnings("ignore")
    model.simulate(simulation_timeline)



FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Coding\\image-materials\\data\\raw\\image\\SSP2_VLHO\\trp_frgt_Tkm.out'

In [ ]:
model.grid["inflow_materials"].to_array()

: 

: 

In [ ]:
# sum inflow materials for steel, sum also per types keep regions and year

materials_dict_metal = {
    'Steel' : 'Steel',
    'Aluminium' : 'Aluminium',
    'Copper' : 'Cu',
}

materials_dict_nmm = {
    'Cement' : 'Cement',
    'Sand' : 'Sand'
}

# Conversion factors
# always taking the lower range numbers to be cautios

# https://civiltoday.com/civil-engineering-materials/cement/10-cement-ingredients-with-functions
# Cement: Lime 60-75%, Silica 17-25%, other aggregates
# https://concretesupplyco.com/concrete-basics/
# Concrete:  10% cement, 20% air and water, 30% sand, and 40% gravel --> 30% + 40% = 70%
# https://samsa.org.uk/key_uses/glass.php, https://www.carmeuse.com/na-en/references/case-studies-success-stories/limestone-glassmaking-what-you-need-know
# Sand (silica) in glass: 70%, lime: 15%

from imagematerials.rest_of.preprocessing.sum_sectors_image_materials import sum_inflows_for_output

total_material_dict_metals = sum_inflows_for_output(model, materials_dict_metal, 'metals', save = True)
total_material_dict_nmm = sum_inflows_for_output(model, materials_dict_nmm, 'nmm', save = True)



: 

: 

In [ ]:

from imagematerials.rest_of.const import REGION_TO_CLASS_DICT_IMAGE_MAT_NR

def sum_inflows_for_output(model_name, materials_dict, resource_group, save = True):
    cement_in_concrete = 0.1
    sand_in_cement_conversion = 0.17 #(silica)
    sand_gravel_in_concrete_conversion = 0.7
    sand_in_glass_conversion = 0.7

    only_buildings = ['Cement', 'Concrete']
    only_vehicles = ['Glass']
    not_in_any = ['Sand']
    total_material_dict = {}

    # regions electricity and generation
    inflow_materials_grid = model_name.grid["inflow_materials"].to_array()
    inflow_materials_generation = model_name.generation["inflow_materials"].to_array()

     # replace region names to numbers
    # replace region names to numbers in the 'Region' coordinate
    inflow_materials_grid = inflow_materials_grid.assign_coords(
        Region = inflow_materials_grid.coords["Region"].to_series().map(REGION_TO_CLASS_DICT_IMAGE_MAT_NR)
    )
    inflow_materials_generation = inflow_materials_generation.assign_coords(
        Region = inflow_materials_generation.coords["Region"].to_series().map(REGION_TO_CLASS_DICT_IMAGE_MAT_NR)
    )

    for key, value in materials_dict.items():
        if key not in only_buildings and key not in only_vehicles and not key in not_in_any:
            inflow_buildings = model_name.buildings.get('inflow_materials').to_array().sum(['Type']).sel(material=key).loc[1961:]
            inflow_vehicles = model_name.vehicles.get('inflow_materials').to_array().sum(['Type']).sel(material=value).loc[1961:]
            inflow_electricity_genereation = inflow_materials_generation.sum(['Type']).sel(material=value).loc[1961:]
            inflow_electricity_grid = inflow_materials_grid.sum(['Type']).sel(material=value).loc[1961:]
            total_material = inflow_vehicles + inflow_electricity_genereation + inflow_electricity_grid + inflow_buildings

        if key == 'Cement':
            # add concrete to cement
            inflow_buildings_cement = model_name.buildings.get('inflow_materials').to_array().sum(['Type']).sel(material=key).loc[1961:]
            inflow_buildings_concrete = model_name.buildings.get('inflow_materials').to_array().sum(['Type']).sel(material='Concrete').loc[1961:] * cement_in_concrete
            inflow_electricity_genereation =inflow_materials_generation.sum(['Type']).sel(material='Concrete').loc[1961:] * cement_in_concrete
            inflow_electricity_grid = inflow_materials_grid.sum(['Type']).sel(material='Concrete').loc[1961:] * cement_in_concrete

            total_material = inflow_electricity_genereation + inflow_electricity_grid + inflow_buildings_cement + inflow_buildings_concrete
        
        if key == 'Sand':
            inflow_buildings_cement_sand = model_name.buildings.get('inflow_materials').to_array().sum(['Type']).sel(material='Cement').loc[1961:]*sand_in_cement_conversion
            inflow_buildings_concrete_sand_via_cement = model_name.buildings.get('inflow_materials').to_array().sum(['Type']).sel(material='Concrete').loc[1961:] * cement_in_concrete * sand_in_cement_conversion
            inflow_buildings_sand_in_concrete = model_name.buildings.get('inflow_materials').to_array().sum(['Type']).sel(material='Concrete').loc[1961:] * sand_gravel_in_concrete_conversion
            inflow_vehicles_sand = model_name.vehicles.get('inflow_materials').to_array().sum(['Type']).sel(material='Glass').loc[1961:] * sand_in_glass_conversion

            inflow_electricity_genereation_sand = inflow_materials_generation.sum(['Type']).sel(material='Glass').loc[1961:] * sand_in_glass_conversion
            inflow_electricity_grid_sand = inflow_materials_grid.sum(['Type']).sel(material='Glass').loc[1961:] * sand_in_glass_conversion

            inflow_electricity_genereation_concrete_sand_via_cement = inflow_materials_generation.sum(['Type']).sel(material='Concrete').loc[1961:] * cement_in_concrete * sand_in_cement_conversion
            inflow_electricity_grid_concrete_sand_via_cement = inflow_materials_grid.sum(['Type']).sel(material='Concrete').loc[1961:] * cement_in_concrete * sand_in_cement_conversion

            total_material = (inflow_buildings_cement_sand + inflow_buildings_concrete_sand_via_cement + inflow_buildings_sand_in_concrete + 
                              inflow_vehicles_sand + inflow_electricity_genereation_sand + inflow_electricity_grid_sand + 
                              inflow_electricity_genereation_concrete_sand_via_cement + inflow_electricity_grid_concrete_sand_via_cement)

        # from total_material create a csv that has the years as rows and regions as columns, mae sure that region names are no just '1' but 'class_ 1'
        # also drop material dimension
        if key not in ['Copper', 'Cement', 'Sand']:
            total_material = total_material.drop_vars('material')
        # change the region coordinate so that it is class_ 1 instead of 1 , ...
        # Get the current region values
        regions = total_material.coords['Region'].values

        # Create new region names
        new_regions = [f'class_ {r}' for r in regions]

        # Assign the new region names to the coordinate
        total_material = total_material.assign_coords(Region=new_regions)
        # to t
        total_material = total_material.pint.to('t')
        # save as pandas to save as csv
        total_material = total_material.rename("total_material")
        # write key with a small letter
        key = key.lower()
        # to pandas
        total_material = total_material.to_dataframe().unstack()
        # drop unessecary column level index
        total_material.columns = total_material.columns.droplevel(0)
        # save as csv
        if key == 'sand':
            key = 'sand_gravel_crushed_rock'
            total_material = total_material.loc[1971:]
        else: 
            pass

        if save == True:
            total_material.to_csv(f'../data/raw/rest-of/{resource_group}/image_materials_{key}.csv')
            print('done', key)

        total_material_dict[key] = total_material

    return total_material_dict, inflow_vehicles, inflow_electricity_genereation, inflow_buildings

total_material_dict_metals, inflow_vehicles_metals, inflow_electricity_genereation_metals, inflow_buildings_metals = sum_inflows_for_output(model, materials_dict_metal, 'metals', save = True)


: 

: 

In [ ]:
inflow_vehicles_metals

: 

: 

In [ ]:
inflow_buildings_metals

: 

: 

In [ ]:
inflow_electricity_genereation_metals + inflow_vehicles_metals + inflow_buildings_metals

: 

: 

In [ ]:
total_material_dict_metals["copper"].columns

: 

: 

In [ ]:
inflow_materials = model.grid["inflow_materials"].to_array().sel(material='Cu').sum(['Type'])
inflow_materials_vehicles = model.vehicles["inflow_materials"].to_array().sel(material='Cu').sum(['Type'])

inflow_materials_vehicles


: 

: 